# Phase S1 — TPU Version: Cooling Theory Validation
## Validate BatchNorm as a cooling mechanism: φ_BN ≈ 0.66

**Simplified for TPU:** 2 configs × 3 D × 1 seed = 6 runs (~3-5h on TPU)

| Config | norm | D values | seeds |
|--------|------|----------|-------|
| None_NoSkip | none | 32, 48, 96 | 42 |
| BN_NoSkip | batchnorm | 32, 48, 96 | 42 |


## Step 1: TPU Setup

In [ ]:
# TPU setup
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

# Verify TPU is accessible
try:
    DEVICE = xm.xla_device()
    test_tensor = torch.zeros((2, 3), device=DEVICE)
    print('TPU device:', DEVICE)
    print('TPU runtime: OK')
except Exception as e:
    raise RuntimeError(
        f'TPU not accessible: {e}. '
        'Go to Notebook Settings → Accelerator → TPU → Save. '
        'Then restart the session (Kernel → Restart & clear output).'
    )


## Step 2: Clone + Install

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret('gh_token')
repo_url = f'https://{gh_token}@github.com/xliu203/ThermoRG-NN.git'
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email 'xliu203@asu.edu'
!git config --global user.name 'Leo Liu'
print('Cloned develop branch')


## Step 3: Imports

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/experiments/phase_s1')

import math, time, json
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from scipy.optimize import curve_fit
from scipy.linalg import svd

import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

print(f'PyTorch: {torch.__version__}')
print(f'TPU: {xm.xla_device()}')


## Step 4: Config + Network Definitions

In [ ]:
# ─── Config ─────────────────────────────────────────────
CONFIGS     = [('None_NoSkip', 'none', [42]), ('BN_NoSkip', 'batchnorm', [42])]
D_VALUES    = [32, 48, 64, 96]
EPOCHS      = 200
BATCH_SIZE  = 256
LR          = 0.01
WD          = 5e-4
MOMENTUM    = 0.9
GAMMA_TRACK = [50, 100, 150, 200]

OUT_DIR  = Path('/kaggle/working/phase_s1_tpu_results')
OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

print(f'Configs: {[c[0] for c in CONFIGS]}')
print(f'D values: {D_VALUES}')
print(f'Epochs: {EPOCHS}')


In [ ]:
# ─── Network ────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, stride=1):
        super().__init__()
        self.norm_type = norm_type
        self.use_skip  = use_skip
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        if norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        else:
            self.norm = nn.Identity()
        self.act = nn.GELU()
        if use_skip and in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        elif use_skip:
            self.skip = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
        else:
            self.skip = None

    def forward(self, x):
        out = self.norm(self.conv(x))
        out = self.act(out)
        if self.use_skip and self.skip is not None:
            out = out + self.skip(x)
        return out


class ValidationNet(nn.Module):
    def __init__(self, base_ch=64, norm_type='none', use_skip=False, n_classes=10):
        super().__init__()
        channels = [3, base_ch, base_ch*2, base_ch*2]
        self.blocks = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.blocks.append(ConvBlock(channels[i], channels[i+1],
                                        norm_type=norm_type,
                                        use_skip=(i > 0 and use_skip),
                                        stride=1))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc   = nn.Linear(channels[-1], n_classes)

    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
# ─── J_topo ─────────────────────────────────────────────────────
def compute_D_eff(W):
    if W.dim() == 4:
        W = W.view(W.size(0), -1)
    fro_sq  = (W ** 2).sum().item()
    S = svd(W.cpu().numpy(), compute_uv=False)
    spec_sq = S[0]**2 + 1e-12
    return fro_sq / spec_sq

def compute_J_topo(weights, d_input=3.0):
    eta_vals, d_prev = [], d_input
    for W in weights:
        if W.dim() == 4:
            W = W.view(W.size(0), -1)
        D_eff = compute_D_eff(W)
        eta = D_eff / max(d_prev, 1e-8)
        eta_vals.append(max(eta, 1e-8))
        d_prev = D_eff
    L = len(eta_vals)
    return math.exp(-sum(abs(math.log(e)) for e in eta_vals) / L) if L > 0 else 0.0


# ─── Variance Tracker ───────────────────────────────────────────
class VarianceTracker:
    def __init__(self, model):
        self.model = model
        self.activations = {}
        self.handles = []
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                h = module.register_forward_hook(
                    lambda mod, inp, out, n=name: self.activations.update({n: out.detach()}))
                self.handles.append(h)

    def get_variances(self):
        return {n: a.var().item() for n, a in self.activations.items()}

    def compute_gamma(self, init_variances):
        final_vars = self.get_variances()
        total, count = 0.0, 0
        for name, vi in init_variances.items():
            if name in final_vars:
                si = math.sqrt(vi)
                sf = math.sqrt(final_vars[name])
                if si > 1e-8 and sf > 1e-8:
                    total += abs(math.log(sf / si))
                    count += 1
        return total / max(count, 1)

    def close(self):
        for h in self.handles:
            h.remove()


In [ ]:
# ─── DataLoaders ──────────────────────────────────────────────────
def get_dataloaders():
    tfm_train = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                           T.ToTensor(), T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
    tfm_val  = T.Compose([T.ToTensor(), T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
    train_ds = CIFAR10(root='./data', train=True,  transform=tfm_train, download=True)
    val_ds   = CIFAR10(root='./data', train=False, transform=tfm_val,  download=True)
    # TPU: num_workers=0 is optimal
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, val_loader


## Step 5: Smoke Test

In [ ]:
x = torch.randn(4, 3, 32, 32)
y = torch.randint(0, 10, (4,))
for name, norm in [('None', 'none'), ('BN', 'batchnorm')]:
    m = ValidationNet(base_ch=32, norm_type=norm, use_skip=False).to(DEVICE)
    out = m(x.to(DEVICE))
    loss = nn.functional.cross_entropy(out, y.to(DEVICE))
    loss.backward()
    print(f'{name}: output {out.shape}, loss {loss.item():.4f} [OK]')
    del m
print('\nSmoke test PASSED')


## Step 6: Training Functions

In [ ]:
def measure_init_variance(model, batch_size=64):
    model.eval()
    tracker = VarianceTracker(model)
    dummy = torch.randn(batch_size, 3, 32, 32).to(DEVICE)
    with torch.no_grad(): model(dummy)
    init_vars = tracker.get_variances()
    tracker.close()
    return init_vars


def fit_scaling_law(losses_by_d, d_values):
    def power_law(D, alpha, beta, E):
        return alpha * np.array(D)**(-beta) + E
    Ds = np.array(d_values)
    losses = np.array([losses_by_d[d] for d in d_values])
    try:
        popt, _ = curve_fit(power_law, Ds, losses, p0=[10, 0.5, 0.5],
                           bounds=([0,0,0],[1000,5,10]), maxfev=10000)
        r2 = 1 - (((losses - power_law(Ds,*popt))**2).sum() /
                    ((losses - losses.mean())**2).sum())
        return {'alpha':float(popt[0]),'beta':float(popt[1]),'E':float(popt[2]),'R2':float(r2)}
    except:
        return {'alpha':None,'beta':None,'E':None,'R2':0.0}


def train_one_run(model, train_loader, val_loader, epochs, lr, wd, momentum,
                  init_variances=None, track_gamma=False, cfg_name='', base_ch=0, seed=0):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    best_loss, best_acc, best_ep = float('inf'), 0.0, 0
    gamma_history = []
    tracker = VarianceTracker(model) if (track_gamma and init_variances) else None
    t0 = time.time()

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        mp_train = pl.MpDeviceLoader(train_loader, DEVICE)
        for X, y in mp_train:
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            xm.optimizer_step(optimizer, barrier=True)
        scheduler.step()

        # ── Eval ──
        model.eval()
        loss_sum, correct, total = 0.0, 0, 0
        mp_val = pl.MpDeviceLoader(val_loader, DEVICE)
        with torch.no_grad():
            for X, y in mp_val:
                out = model(X)
                loss_sum += criterion(out, y).item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)
        val_loss = loss_sum / total
        val_acc  = correct / total

        if val_loss < best_loss:
            best_loss, best_acc, best_ep = val_loss, val_acc, epoch+1

        # ── Gamma tracking ──
        if tracker and ((epoch+1) in GAMMA_TRACK or epoch == epochs-1):
            model.eval()
            with torch.no_grad(): model(torch.randn(64,3,32,32).to(DEVICE))
            gamma_history.append({'epoch':epoch+1, 'gamma': tracker.compute_gamma(init_variances)})

        if (epoch+1) % 25 == 0 or epoch == epochs-1:
            elapsed = (time.time()-t0)/60
            print(f'  [{cfg_name}] ch={base_ch} s={seed} ep={epoch+1:3d} '
                  f'loss={val_loss:.4f} acc={val_acc:.4f} best={best_loss:.4f} [{elapsed:.1f}m]')

    if tracker: tracker.close()
    return {'best_val_loss':best_loss,'best_val_acc':best_acc,'best_epoch':best_ep,
            'gamma_history':gamma_history}


## Step 7: Run Full Training

In [ ]:
from datetime import datetime
print('\n' + '='*70)
print('STARTING Phase S1 TPU — Cooling Theory Validation')
print('='*70)

train_loader, val_loader = get_dataloaders()
print(f'Data loaded. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}')

meta_file = OUT_DIR / 'metadata.json'
if meta_file.exists():
    with open(meta_file) as f: metadata = json.load(f)
else:
    metadata = {'completed_runs': []}
completed = set(metadata['completed_runs'])

total_runs = sum(len(c[2]) * len(D_VALUES) for c in CONFIGS)
print(f'Total runs: {total_runs} ({len(completed)} done)')
print(f'Epochs: {EPOCHS} | D: {D_VALUES}')
print('-'*60)

all_results = {'timestamp': datetime.now().isoformat(), 'epochs': EPOCHS, 'configs': []}
t_start = time.time()

for cfg_name, norm_type, seeds in CONFIGS:
    cfg_start = time.time()
    # J_topo at init
    m_init = ValidationNet(base_ch=64, norm_type=norm_type, use_skip=False).to(DEVICE)
    J_topo = compute_J_topo(m_init.get_conv_weights())
    del m_init
    cfg_result = {'name':cfg_name,'norm':norm_type,'J_topo_init':float(J_topo),'D_results':{}}

    for base_ch in D_VALUES:
        n_params = (3*base_ch*9 + 2*base_ch*base_ch*9*2 + 2*base_ch*10) / 1e6
        d_result = {'base_ch':base_ch,'n_params_M':float(n_params),
                    'seeds':{},'avg_val_loss':None,'avg_gamma':None}
        losses, gammas = [], []

        for seed in seeds:
            run_name = f'{cfg_name}_ch{base_ch}_s{seed}'
            ckpt_path = CKPT_DIR / f'{run_name}.pt'

            if run_name in completed and ckpt_path.exists():
                ckpt = torch.load(ckpt_path, map_location=DEVICE)
                result = ckpt['result']
                losses.append(result['best_val_loss'])
                if result.get('gamma_history'):
                    gammas.append(np.mean([g['gamma'] for g in result['gamma_history']]))
                d_result['seeds'][str(seed)] = result
                print(f'  [{cfg_name}] ch={base_ch} s={seed} [SKIP]')
                continue

            print(f'\n  ▶ [{cfg_name}] ch={base_ch} s={seed} ({n_params:.1f}M)...')
            torch.manual_seed(seed)
            model = ValidationNet(base_ch=base_ch, norm_type=norm_type, use_skip=False).to(DEVICE)
            init_vars = measure_init_variance(model, batch_size=64)
            print(f'    init avg variance: {np.mean(list(init_vars.values())):.4f}')

            t0 = time.time()
            result = train_one_run(model, train_loader, val_loader,
                epochs=EPOCHS, lr=LR, wd=WD, momentum=MOMENTUM,
                init_variances=init_vars, track_gamma=True,
                cfg_name=cfg_name, base_ch=base_ch, seed=seed)
            elapsed = (time.time()-t0)/60

            torch.save({'epoch':EPOCHS-1,'model_state':model.state_dict(),'result':result}, ckpt_path)
            avg_gamma = np.mean([g['gamma'] for g in result['gamma_history']]) if result['gamma_history'] else 0
            print(f'    → loss={result["best_val_loss"]:.4f} acc={result["best_val_acc"]:.4f} '
                  f'γ={avg_gamma:.3f} [{elapsed:.1f}m]')

            gammas.append(avg_gamma)
            losses.append(result['best_val_loss'])
            d_result['seeds'][str(seed)] = result
            completed.add(run_name)
            metadata['completed_runs'] = list(completed)
            with open(meta_file, 'w') as f: json.dump(metadata, f)
            del model

        # ── Save D result (indented INSIDE for base_ch loop) ──
        if losses:
            d_result['avg_val_loss'] = float(np.mean(losses))
            d_result['avg_gamma']    = float(np.mean(gammas)) if gammas else None
        cfg_result['D_results'][str(base_ch)] = d_result

    # ── Scaling law fit (indented inside for cfg_name loop) ──
    losses_by_d = {ch: cfg_result['D_results'][str(ch)]['avg_val_loss']
                   for ch in D_VALUES
                   if cfg_result['D_results'][str(ch)]['avg_val_loss'] is not None}
    if losses_by_d:
        fit = fit_scaling_law(losses_by_d, list(losses_by_d.keys()))
        cfg_result['scaling_fit'] = fit
        wall = (time.time()-cfg_start)/60
        print(f'\n  [{cfg_name}] FIT: β={fit.get("beta",0):.4f} R²={fit.get("R2",0):.4f} [{wall:.0f}m]')

    cfg_result['wall_time_min'] = (time.time()-cfg_start)/60
    all_results['configs'].append(cfg_result)

out_file = OUT_DIR / 'phase_s1_tpu_results.json'
with open(out_file, 'w') as f: json.dump(all_results, f, indent=2)

total_time = (time.time()-t_start)/60
print(f'\n{"="*70}')
print(f'ALL DONE in {total_time:.1f} min')
print(f'Results: {out_file}')


## Step 8: Analysis

In [ ]:
print('\n' + '='*70)
print('SUMMARY')
print('='*70)

base_beta = None
none_cfg = next((c for c in all_results['configs'] if c['name']=='None_NoSkip'), None)
if none_cfg and none_cfg.get('scaling_fit',{}).get('beta'):
    base_beta = none_cfg['scaling_fit']['beta']

print(f"\n{'Config':<15} {'J_topo':<10} {'β':<10} {'φ(β)':<10} {'γ':<10}")
print('-'*60)
for cfg in all_results['configs']:
    fit  = cfg.get('scaling_fit', {})
    beta = fit.get('beta') or 0
    phi  = beta/base_beta if base_beta and base_beta>0 else 0
    gammas = [cfg['D_results'][str(ch)]['avg_gamma']
              for ch in D_VALUES
              if cfg['D_results'][str(ch)]['avg_gamma'] is not None]
    avg_g = sum(gammas)/len(gammas) if gammas else 0
    print(f"{cfg['name']:<15} {cfg.get('J_topo_init',0):<10.4f} "
          f"{beta:<10.4f} {phi:<10.3f} {avg_g:<10.4f}")

print('\n' + '='*70)
print('COOLING ANALYSIS')
print('='*70)
if base_beta:
    bn_cfg = next((c for c in all_results['configs'] if c['name']=='BN_NoSkip'), None)
    if bn_cfg and bn_cfg.get('scaling_fit',{}).get('beta'):
        phi_bn = bn_cfg['scaling_fit']['beta'] / base_beta
        print(f'  φ_BN = {phi_bn:.3f}  (theory: ≈ 0.66)')
        print(f'  → If φ_BN ≈ 1.0: no cooling effect')
        print(f'  → If φ_BN ≈ 0.66: BatchNorm reduces β (cooling confirmed)')
        print(f'  → If φ_BN > 1.0: BatchNorm enhances β (heating)')


## Step 9: Push to GitHub

In [ ]:
OUT_DIR  = '/kaggle/working/phase_s1_tpu_results'
CKPT_DIR = f'{OUT_DIR}/checkpoints'
cmds = [
    f'git add {OUT_DIR}/*.json 2>/dev/null || true',
    f'git add {CKPT_DIR}/*.pt 2>/dev/null || true',
    "git add experiments/phase_s1/notebooks/*.ipynb 2>/dev/null || true",
    "git commit -m 'Data: Phase S1 cooling theory TPU validation' || true",
    'git push origin develop 2>&1 || true',
]
for cmd in cmds:
    !$cmd
print('Push done')
